# Predicción de accidentalidad — El Poblado

Notebook de partida para la parte de leaderboard. El EDA, la calidad de datos y el resto de secciones del taller van en su propio notebook/informe — ver el PDF del taller y el README de este assignment.

**Antes de empezar, lean la sección "La trampa principal" del README.**

In [ ]:
import sqlite3

import pandas as pd

DB_PATH = "data_accidentes_poblado.sqlite3"  # ruta al archivo que descargaron
TEST_START = pd.Timestamp("2019-08-01 00:00:00")
TEST_END = pd.Timestamp("2019-12-31 00:00:00")

con = sqlite3.connect(DB_PATH)
clima = pd.read_sql("SELECT * FROM clima", con, parse_dates=["TW"])
accidentes = pd.read_sql("SELECT * FROM accidentes", con, parse_dates=["TW"])
raw = pd.read_sql("SELECT * FROM raw_accidentes", con, parse_dates=["TW"])
con.close()

print("clima:", clima.shape, clima.TW.min(), "->", clima.TW.max())
print("accidentes:", accidentes.shape, "| ultimo:", accidentes.TW.max())
print("barrios:", clima.BARRIO.nunique())

## Construir el target

`clima` es el universo de parejas (barrio, hora). El target vale 1 si esa pareja aparece en `accidentes`, 0 si no.

In [ ]:
positivos = set(zip(accidentes["BARRIO"], accidentes["TW"]))
clima["target"] = [
    1 if par in positivos else 0
    for par in zip(clima["BARRIO"], clima["TW"])
]

train = clima[clima["TW"] < TEST_START].copy()
test = clima[(clima["TW"] >= TEST_START) & (clima["TW"] < TEST_END)].copy()

print("train:", train.shape, "| tasa positivos:", round(train["target"].mean(), 4))
print("test (a predecir):", test.shape)
# Ojo: test['target'] es 0 en todas las filas por construcción -- NO lo usen.

## Validación temporal

Para estimar honestamente su desempeño, reserven los últimos meses del período de entrenamiento como validación — imitando la relación que hay entre train y el período de evaluación real.

In [ ]:
VAL_START = pd.Timestamp("2019-04-01")
sub_train = train[train["TW"] < VAL_START]
val = train[train["TW"] >= VAL_START]
print("sub-train:", len(sub_train), "| validacion:", len(val))

## Ingeniería de características

TODO. Ideas del taller (sección 4.3): variables de calendario, codificación cíclica seno/coseno, ventanas móviles de clima, agregados por barrio.

Recuerden: los agregados **de accidentes** deben calcularse con corte al 2019-07-31 (ver README). Los de **clima** sí pueden usar todo el período.

In [ ]:
# TODO: features

## Modelado

TODO: estrategias de balanceo, comparación de familias de modelos, ajuste de hiperparámetros.

Evalúen con `average_precision_score`, que es la métrica del leaderboard.

In [ ]:
# from sklearn.metrics import average_precision_score
# print(average_precision_score(y_val, modelo.predict_proba(X_val)[:, 1]))

## Generar la submission

In [ ]:
# proba = modelo.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test["BARRIO"] + "|" + test["TW"].dt.strftime("%Y-%m-%d %H:%M:%S"),
    "target": 0.0,  # reemplazar por `proba`
})
submission.to_csv("mi_prediccion.csv", index=False)
print(len(submission), "filas")  # debe ser 80256
submission.head()